01: “Baseline: TF-IDF + Logistic Regression. Accuracy 86.3%, macro F1 0.80. Weak on love and surprise (recall 0.60 / 0.47).”

In [1]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

In [2]:
from google.colab import files
uploaded = files.upload()

Saving train.txt to train.txt


In [3]:
df = pd.read_csv('train.txt', sep=';', header=None, names=['text', 'emotion'])

In [4]:
df.head(4)

,text,emotion
0,i didnt feel humiliated,sadness
1,i can go from feeling so hopeless to so damned...,sadness
2,im grabbing a minute to post i feel greedy wrong,anger
3,i am ever feeling nostalgic about the fireplac...,love


In [5]:
df.isnull().sum()

,0
text,0
emotion,0


In [6]:
unique_emotions = df['emotion'].unique()
emotions_numbers = {}
i = 0
for emo in unique_emotions:
  i +=1
  emotions_numbers[emo] = i
print(i)

df['emotion'] = df['emotion'].map(emotions_numbers)



6


In [7]:
df

,text,emotion
0,i didnt feel humiliated,1
1,i can go from feeling so hopeless to so damned...,1
2,im grabbing a minute to post i feel greedy wrong,2
3,i am ever feeling nostalgic about the fireplac...,3
4,i am feeling grouchy,2
...,...,...
15995,i just had a very brief time in the beanbag an...,1
15996,i am now turning and i feel pathetic that i am...,1
15997,i feel strong and good overall,6
15998,i feel like this was such a rude comment and i...,2


In [8]:
df['text'] = df['text'].apply(lambda x : x.lower())

In [9]:
df

,text,emotion
0,i didnt feel humiliated,1
1,i can go from feeling so hopeless to so damned...,1
2,im grabbing a minute to post i feel greedy wrong,2
3,i am ever feeling nostalgic about the fireplac...,3
4,i am feeling grouchy,2
...,...,...
15995,i just had a very brief time in the beanbag an...,1
15996,i am now turning and i feel pathetic that i am...,1
15997,i feel strong and good overall,6
15998,i feel like this was such a rude comment and i...,2


In [10]:
import string
def remove_punc(txt):
  return txt.translate(str.maketrans('','',string.punctuation))

In [11]:
df['text'] = df['text'].apply(remove_punc)

In [12]:
df['text']

,text
0,i didnt feel humiliated
1,i can go from feeling so hopeless to so damned...
2,im grabbing a minute to post i feel greedy wrong
3,i am ever feeling nostalgic about the fireplac...
4,i am feeling grouchy
...,...
15995,i just had a very brief time in the beanbag an...
15996,i am now turning and i feel pathetic that i am...
15997,i feel strong and good overall
15998,i feel like this was such a rude comment and i...


In [13]:
def remove_num(txt):
  new = ""
  for i in txt:
    if not i.isdigit():
      new = new + i
  return new

df['text'] = df['text'].apply(remove_num)

In [14]:
def remove_emoji(txt):
  rm_emoji = ""
  for i in txt:
    if i.isascii():
      rm_emoji = rm_emoji + i
  return rm_emoji
df['text'] = df['text'].apply(remove_emoji)

In [15]:
import nltk

In [16]:
from nltk.corpus import stopwords
from nltk.tokenize import word_tokenize

In [17]:
nltk.download('punkt')
nltk.download('stopwords')


[nltk_data] Downloading package punkt to /root/nltk_data...
[nltk_data]   Unzipping tokenizers/punkt.zip.
[nltk_data] Downloading package stopwords to /root/nltk_data...
[nltk_data]   Unzipping corpora/stopwords.zip.


True

In [18]:
stop_words = set(stopwords.words('english'))

In [19]:
stop_words

{'a',
 'about',
 'above',
 'after',
 'again',
 'against',
 'ain',
 'all',
 'am',
 'an',
 'and',
 'any',
 'are',
 'aren',
 "aren't",
 'as',
 'at',
 'be',
 'because',
 'been',
 'before',
 'being',
 'below',
 'between',
 'both',
 'but',
 'by',
 'can',
 'couldn',
 "couldn't",
 'd',
 'did',
 'didn',
 "didn't",
 'do',
 'does',
 'doesn',
 "doesn't",
 'doing',
 'don',
 "don't",
 'down',
 'during',
 'each',
 'few',
 'for',
 'from',
 'further',
 'had',
 'hadn',
 "hadn't",
 'has',
 'hasn',
 "hasn't",
 'have',
 'haven',
 "haven't",
 'having',
 'he',
 "he'd",
 "he'll",
 "he's",
 'her',
 'here',
 'hers',
 'herself',
 'him',
 'himself',
 'his',
 'how',
 'i',
 "i'd",
 "i'll",
 "i'm",
 "i've",
 'if',
 'in',
 'into',
 'is',
 'isn',
 "isn't",
 'it',
 "it'd",
 "it'll",
 "it's",
 'its',
 'itself',
 'just',
 'll',
 'm',
 'ma',
 'me',
 'mightn',
 "mightn't",
 'more',
 'most',
 'mustn',
 "mustn't",
 'my',
 'myself',
 'needn',
 "needn't",
 'no',
 'nor',
 'not',
 'now',
 'o',
 'of',
 'off',
 'on',
 'once',
 'on

In [20]:
len(stop_words)

198

In [21]:
df.loc[1]['text']

'i can go from feeling so hopeless to so damned hopeful just from being around someone who cares and is awake'

In [22]:
def remove(txt):
  words = txt.split()
  cleaned = []
  for i in words:
    if not i in stop_words:
      cleaned.append(i)
  return ' '.join(cleaned)


In [23]:
df['text'] = df['text'].apply(remove)

In [24]:
df.loc[1]['text']

'go feeling hopeless damned hopeful around someone cares awake'

In [25]:
 from sklearn.model_selection import train_test_split

 X_train, X_test, y_train, y_test = train_test_split(
     df['text'], df['emotion'], test_size=0.20, random_state=42)

In [26]:
df.head(4)

,text,emotion
0,didnt feel humiliated,1
1,go feeling hopeless damned hopeful around some...,1
2,im grabbing minute post feel greedy wrong,2
3,ever feeling nostalgic fireplace know still pr...,3


In [27]:
df.shape

(16000, 2)

In [28]:
from sklearn.feature_extraction.text import TfidfVectorizer , CountVectorizer

In [29]:
bow_vectorizer = CountVectorizer()
X_train_bow = bow_vectorizer.fit_transform(X_train)
X_test_bow = bow_vectorizer.transform(X_test)

tfidf_vectorizer = TfidfVectorizer()

In [30]:
from sklearn.naive_bayes import MultinomialNB
from sklearn.metrics import accuracy_score

In [31]:
nb_model = MultinomialNB()

In [32]:
X_train_bow = bow_vectorizer.fit_transform(X_train)
X_test_bow = bow_vectorizer.transform(X_test)

nb_model.fit(X_train_bow, y_train)
nb_model

MultinomialNB()

In [33]:
pred_nb = nb_model.predict(X_test_bow)
print(accuracy_score(y_test, pred_nb))

0.768125


In [34]:
pred_nb

array([1, 6, 1, ..., 6, 6, 1])

In [35]:
y_test

,emotion
8756,1
4660,6
6095,1
304,6
8241,1
...,...
15578,6
5746,6
6395,6
7624,6


In [36]:
X_train_tfidf = tfidf_vectorizer.fit_transform(X_train)
X_test_tfidf = tfidf_vectorizer.transform(X_test)

tfidf_vectorizer = TfidfVectorizer()

nb2_model = MultinomialNB()
nb2_model.fit(X_train_tfidf, y_train)

MultinomialNB()

In [37]:
y_pred = nb2_model.predict(X_test_tfidf)

In [38]:
print(accuracy_score(y_test, y_pred))

0.6609375


In [39]:
from sklearn.linear_model import LogisticRegression

In [40]:
logistic_model = LogisticRegression(max_iter=1000)

In [41]:
logistic_model.fit(X_train_tfidf, y_train)

LogisticRegression(max_iter=1000)

In [42]:
log_pred = logistic_model.predict(X_test_tfidf)

In [43]:
print(accuracy_score(y_test, log_pred))

0.8628125
